# GPT2-small CPT on SID-v2

Canonical CPT recipe for the new Qwen3-Embedding-4B + RQ-VAE Semantic IDs.

Mixture design:
- 50% train-only behavior histories.
- 50% item metadata grounding.
- Inside metadata: 70% compact metadata (`title + year + genres`) and 30% short plot overview.

This notebook is intentionally not executed. Run cells manually when you are ready.


Precision policy:
- use `bf16` on setups where PyTorch reports BF16 support;
- otherwise use `fp16` on CUDA;
- never enable `bf16` and `fp16` together, because HuggingFace `TrainingArguments` treats them as mutually exclusive mixed-precision modes.


## 1. Imports and config

The sequence length is capped to keep GPT2-small CPT fast. The behavior side reads only `data/processed/splits/train.parquet`; validation/test recommendation splits are not used as behavior input.


In [11]:
from pathlib import Path
import inspect
import json
import random

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset

from transformers import (
    DataCollatorForLanguageModeling,
    GPT2LMHeadModel,
    GPT2TokenizerFast,
    Trainer,
    TrainingArguments,
)


In [12]:
from importlib import metadata
from packaging.version import Version

try:
    accelerate_version = metadata.version("accelerate")
except metadata.PackageNotFoundError as exc:
    raise ImportError(
        "This notebook needs accelerate for HuggingFace Trainer. "
        "Install it in this exact kernel with: %pip install -U accelerate"
    ) from exc

if Version(accelerate_version) < Version("0.21.0"):
    raise ImportError(
        f"accelerate>={0.21} is required, found {accelerate_version}. "
        "Upgrade in this exact kernel with: %pip install -U accelerate"
    )

print("accelerate:", accelerate_version)


accelerate: 1.13.0


In [22]:
ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

BASE_MODEL = "gpt2"  # GPT2-small
RUN_NAME = "cpt_gpt2_small_sid_v2_qwen4b_plum_m50_b50_descshort"

TRAIN_PATH = ROOT / "data/processed/splits/train.parquet"
USERS_PATH = ROOT / "data/raw/ml-1m/users.dat"
ITEM_PROFILE_PATH = ROOT / "data/processed/item_features/qwen4b_audited_v1_item_profiles.parquet"
SID_ARRAY_PATH = ROOT / "runs/qwen4b_rqvae_sid_v2_plum/SIDs_best.npy"
SID_MAPPING_PATH = ROOT / "runs/qwen4b_rqvae_sid_v2_plum/sid_mapping_best.parquet"

ARTIFACT_DIR = ROOT / "data/processed/artifacts" / RUN_NAME
MODEL_OUT_DIR = ARTIFACT_DIR / "model"
FINAL_MODEL_DIR = ARTIFACT_DIR / "final"

SEED = 42
MAX_SEQ_LENGTH = 384
HISTORY_LAST_K = 48
DESC_TEXT_MAX_TOKENS = 96
MAX_TOTAL_EXAMPLES = 10_000
VALIDATION_SIZE = 0.02

BEHAVIOR_RATIO = 0.50
METADATA_COMPACT_RATIO = 0.70
METADATA_DESCRIPTION_RATIO = 0.30
DESCRIPTION_MODE = "short"  # "none" | "short"

INCLUDE_RATINGS = True
INCLUDE_USER_FEATURES = True

NUM_TRAIN_EPOCHS = 3.0
DRY_RUN_STEPS = 0  # set to 1 for a quick trainer smoke test, then set back to 0 for full CPT
LEARNING_RATE = 5e-5
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.01
PER_DEVICE_BATCH = 8
GRADIENT_ACCUMULATION_STEPS = 4
EVAL_STEPS = 200
SAVE_STEPS = 200
LOGGING_STEPS = 50

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BF16 = DEVICE == "cuda" and torch.cuda.is_bf16_supported()
FP16 = DEVICE == "cuda" and not BF16
PRECISION_MODE = "bf16" if BF16 else ("fp16" if FP16 else "fp32")
if DEVICE == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print(ROOT)
print(DEVICE, PRECISION_MODE)


C:\Users\User\plum-ml1m-repro
cuda bf16


## 2. Load train histories, item profiles, and SID-v2

Behavior examples are built only from the train split. Item metadata comes from the audited Qwen3-Embedding-4B feature stage. SID tokens come from the best RQ-VAE checkpoint.


In [14]:
train = pd.read_parquet(TRAIN_PATH)
users = pd.read_csv(
    USERS_PATH,
    sep="::",
    engine="python",
    names=["user_id", "gender", "age", "occupation", "zip"],
    encoding="latin-1",
)
profiles = pd.read_parquet(ITEM_PROFILE_PATH).sort_values("item_idx").reset_index(drop=True)
sid_mapping = pd.read_parquet(SID_MAPPING_PATH).sort_values("item_idx").reset_index(drop=True)
sids = np.load(SID_ARRAY_PATH)

assert train[["user_id", "user_idx", "item_idx", "rating", "timestamp"]].notna().all().all()
assert np.array_equal(profiles["item_idx"].to_numpy(), np.arange(len(profiles)))
assert np.array_equal(sid_mapping["item_idx"].to_numpy(), np.arange(len(sid_mapping)))
assert sids.shape[0] == len(profiles)
assert sids.shape[1] == 4

print("train rows:", len(train))
print("users:", len(users))
print("items:", len(profiles))
print("sids:", sids.shape)
profiles[["item_idx", "movie_id", "title", "year", "genres_text", "description_text"]].head()


train rows: 988129
users: 6040
items: 3706
sids: (3706, 4)


,item_idx,movie_id,title,year,genres_text,description_text
0,0,1,Toy Story (1995),1995,"Animation, Children's, Comedy","Plot overview: Led by Woody, Andy's toys live ..."
1,1,2,Jumanji (1995),1995,"Adventure, Children's, Fantasy",Plot overview: When siblings Judy and Peter di...
2,2,3,Grumpier Old Men (1995),1995,"Comedy, Romance",Plot overview: A family wedding reignites the ...
3,3,4,Waiting to Exhale (1995),1995,"Comedy, Drama","Plot overview: Cheated on, mistreated and step..."
4,4,5,Father of the Bride Part II (1995),1995,Comedy,Plot overview: Just when George Banks has reco...


## 3. Tokenizer and schema tokens

SID tokens use the existing project convention: `<sid_level_code>`, for example `<sid_0_32>`.


In [15]:
BOS = "<bos>"
EOS = "<eos>"
PAD = "<pad>"
UNK = "<unk>"
USER_OPEN = "<user>"
USER_CLOSE = "</user>"
HIST = "<hist>"
EVENT_OPEN = "<e>"
EVENT_CLOSE = "</e>"
META_OPEN = "<meta>"
META_CLOSE = "</meta>"
DESC_OPEN = "<desc>"
DESC_CLOSE = "</desc>"

SCHEMA_TOKENS = [
    USER_OPEN,
    USER_CLOSE,
    HIST,
    EVENT_OPEN,
    EVENT_CLOSE,
    META_OPEN,
    META_CLOSE,
    DESC_OPEN,
    DESC_CLOSE,
]


def sid_tokens(sid):
    return [f"<sid_{level}_{int(code)}>" for level, code in enumerate(sid)]


def rating_token(rating):
    return f"<rat_{int(rating)}>"


def user_tokens(row):
    return [f"<gen_{row.gender}>", f"<age_{int(row.age)}>", f"<occ_{int(row.occupation)}>"]


tokenizer = GPT2TokenizerFast.from_pretrained(BASE_MODEL)
tokenizer.add_special_tokens({
    "bos_token": BOS,
    "eos_token": EOS,
    "pad_token": PAD,
    "unk_token": UNK,
})

extra_tokens = set(SCHEMA_TOKENS)
extra_tokens.update(rating_token(r) for r in range(1, 6))
extra_tokens.update(token for sid in sids for token in sid_tokens(sid))

if INCLUDE_USER_FEATURES:
    extra_tokens.update(f"<gen_{v}>" for v in users["gender"].dropna().unique())
    extra_tokens.update(f"<age_{int(v)}>" for v in users["age"].dropna().unique())
    extra_tokens.update(f"<occ_{int(v)}>" for v in users["occupation"].dropna().unique())

n_added = tokenizer.add_tokens(sorted(extra_tokens))
print("vocab size:", len(tokenizer), "new tokens:", n_added)


vocab size: 50638 new tokens: 377


## 4. Text helpers

Descriptions are shortened before tokenization. This keeps the metadata part useful but prevents CPT from turning into long plot-summary modeling.


In [16]:
def sid_text(item_idx):
    return " ".join(sid_tokens(sids[int(item_idx)]))


def encode_text(text, max_length=MAX_SEQ_LENGTH):
    return tokenizer.encode(
        text,
        add_special_tokens=False,
        truncation=True,
        max_length=max_length,
    )


def shorten_text(text, max_tokens=DESC_TEXT_MAX_TOKENS):
    ids = tokenizer.encode(str(text), add_special_tokens=False)
    return tokenizer.decode(ids[:max_tokens]).strip()


def nonempty(value):
    return isinstance(value, str) and value.strip() and value.strip().lower() != "nan"


## 5. Build metadata examples

Compact metadata is the main grounding signal. Short descriptions are a smaller auxiliary grounding signal.


In [17]:
compact_meta_examples = []
description_examples = []

for row in profiles.itertuples(index=False):
    compact = (
        f"{BOS} {META_OPEN} "
        f"Movie {sid_text(row.item_idx)} has title: {row.title_clean}. "
        f"Release year: {row.year_text}. "
        f"Genres: {row.genres_text}. "
        f"{META_CLOSE} {EOS}"
    )
    compact_meta_examples.append(encode_text(compact))

    if DESCRIPTION_MODE != "none" and nonempty(row.description_text):
        desc = shorten_text(row.description_text)
        text = (
            f"{BOS} {DESC_OPEN} "
            f"Movie {sid_text(row.item_idx)} plot overview: {desc} "
            f"{DESC_CLOSE} {EOS}"
        )
        description_examples.append(encode_text(text))

print("compact metadata:", len(compact_meta_examples))
print("short descriptions:", len(description_examples))
print("compact length p95:", int(np.percentile([len(x) for x in compact_meta_examples], 95)))
print("description length p95:", int(np.percentile([len(x) for x in description_examples], 95)))


compact metadata: 3706
short descriptions: 3706
compact length p95: 43
description length p95: 115


## 6. Build train-only behavior examples

Each user contributes one behavior sequence built from the latest train events only. Old events are dropped when needed so the sequence stays under `MAX_SEQ_LENGTH`.


In [18]:
users_by_id = users.set_index("user_id", drop=False)
train_sorted = train.sort_values(["user_idx", "timestamp"]).reset_index(drop=True)


def behavior_event(row):
    tokens = [EVENT_OPEN]
    tokens.extend(sid_tokens(sids[int(row.item_idx)]))
    if INCLUDE_RATINGS:
        tokens.append(rating_token(row.rating))
    tokens.append(EVENT_CLOSE)
    return tokens


def fit_behavior(prefix, events):
    selected = list(events[-HISTORY_LAST_K:])
    while selected:
        tokens = prefix + [token for event in selected for token in event] + [EOS]
        ids = tokenizer.convert_tokens_to_ids(tokens)
        if len(ids) <= MAX_SEQ_LENGTH:
            return ids
        selected = selected[1:]
    return tokenizer.convert_tokens_to_ids(prefix + [EOS])


behavior_examples = []
for user_id, user_group in train_sorted.groupby("user_id", sort=False):
    prefix = [BOS]
    if INCLUDE_USER_FEATURES and user_id in users_by_id.index:
        prefix.extend([USER_OPEN, *user_tokens(users_by_id.loc[user_id]), USER_CLOSE])
    prefix.append(HIST)

    events = [behavior_event(row) for row in user_group.itertuples(index=False)]
    behavior_examples.append(fit_behavior(prefix, events))

print("behavior examples:", len(behavior_examples))
print("behavior length p50/p95/max:", np.percentile([len(x) for x in behavior_examples], [50, 95, 100]).astype(int).tolist())


behavior examples: 6040
behavior length p50/p95/max: [344, 344, 344]


## 7. Build the 50/50 CPT mixture

The final corpus is count-balanced: half behavior, half metadata. Inside metadata, compact examples dominate and short descriptions are sampled as auxiliary grounding.


In [19]:
rng = np.random.default_rng(SEED)


def sample_rows(rows, n):
    rows = list(rows)
    replace = n > len(rows)
    idx = rng.choice(len(rows), size=n, replace=replace)
    return [rows[int(i)] for i in idx]


if DESCRIPTION_MODE == "none" or len(description_examples) == 0:
    compact_ratio = 1.0
    desc_ratio = 0.0
else:
    compact_ratio = METADATA_COMPACT_RATIO
    desc_ratio = METADATA_DESCRIPTION_RATIO

metadata_capacity = len(compact_meta_examples) / compact_ratio
if desc_ratio > 0:
    metadata_capacity = min(metadata_capacity, len(description_examples) / desc_ratio)

max_total = int(min(
    len(behavior_examples) / BEHAVIOR_RATIO,
    metadata_capacity / (1.0 - BEHAVIOR_RATIO),
))
max_total = min(max_total, MAX_TOTAL_EXAMPLES)

n_behavior = int(max_total * BEHAVIOR_RATIO)
n_metadata = max_total - n_behavior
n_compact = int(n_metadata * compact_ratio)
n_desc = n_metadata - n_compact

mixed_rows = []
mixed_rows += [{"input_ids": x, "source": "behavior"} for x in sample_rows(behavior_examples, n_behavior)]
mixed_rows += [{"input_ids": x, "source": "metadata_compact"} for x in sample_rows(compact_meta_examples, n_compact)]
if n_desc > 0:
    mixed_rows += [{"input_ids": x, "source": "metadata_description"} for x in sample_rows(description_examples, n_desc)]

rng.shuffle(mixed_rows)
val_n = max(1, int(len(mixed_rows) * VALIDATION_SIZE))
val_rows = mixed_rows[:val_n]
train_rows = mixed_rows[val_n:]

mixture_stats = {
    "total": len(mixed_rows),
    "train": len(train_rows),
    "validation": len(val_rows),
    "behavior": n_behavior,
    "metadata_total": n_metadata,
    "metadata_compact": n_compact,
    "metadata_description": n_desc,
    "max_seq_length": MAX_SEQ_LENGTH,
    "history_last_k": HISTORY_LAST_K,
    "description_mode": DESCRIPTION_MODE,
    "desc_text_max_tokens": DESC_TEXT_MAX_TOKENS,
}
mixture_stats


{'total': 10000,
 'train': 9800,
 'validation': 200,
 'behavior': 5000,
 'metadata_total': 5000,
 'metadata_compact': 3500,
 'metadata_description': 1500,
 'max_seq_length': 384,
 'history_last_k': 48,
 'description_mode': 'short',
 'desc_text_max_tokens': 96}

## 8. Save CPT artifacts

This saves the tokenizer, SID array, corpus tensors, and manifest. The next cell trains from these artifacts.


In [20]:
tokenizer.save_pretrained(ARTIFACT_DIR / "tokenizer")
np.save(ARTIFACT_DIR / "SIDs_best.npy", sids)

corpus_payload = {
    "train_rows": train_rows,
    "val_rows": val_rows,
    "mixture_stats": mixture_stats,
}
torch.save(corpus_payload, ARTIFACT_DIR / "cpt_corpus.pt")

manifest = {
    "run_name": RUN_NAME,
    "base_model": BASE_MODEL,
    "train_path": str(TRAIN_PATH),
    "item_profile_path": str(ITEM_PROFILE_PATH),
    "sid_array_path": str(SID_ARRAY_PATH),
    "sid_mapping_path": str(SID_MAPPING_PATH),
    "uses_recsys_val_test_as_behavior": False,
    "mixture_stats": mixture_stats,
}
with (ARTIFACT_DIR / "manifest.json").open("w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print(ARTIFACT_DIR)
print(json.dumps(mixture_stats, indent=2))


C:\Users\User\plum-ml1m-repro\data\processed\artifacts\cpt_gpt2_small_sid_v2_qwen4b_plum_m50_b50_descshort
{
  "total": 10000,
  "train": 9800,
  "validation": 200,
  "behavior": 5000,
  "metadata_total": 5000,
  "metadata_compact": 3500,
  "metadata_description": 1500,
  "max_seq_length": 384,
  "history_last_k": 48,
  "description_mode": "short",
  "desc_text_max_tokens": 96
}


## 9. Train GPT2-small CPT

Run this only after inspecting the corpus stats. This config is modest: GPT2-small, sequence length 384, batch 8, gradient accumulation 4.


For debugging, set `DRY_RUN_STEPS = 1` in the config cell and run through the train cell. After that works, set it back to `0` for the full 3-epoch CPT.


In [23]:
class LMListDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        return {"input_ids": torch.tensor(self.rows[idx]["input_ids"], dtype=torch.long)}


payload = torch.load(ARTIFACT_DIR / "cpt_corpus.pt", weights_only=False)
train_ds = LMListDataset(payload["train_rows"])
val_ds = LMListDataset(payload["val_rows"])

tokenizer = GPT2TokenizerFast.from_pretrained(ARTIFACT_DIR / "tokenizer")
model = GPT2LMHeadModel.from_pretrained(BASE_MODEL)
model.resize_token_embeddings(len(tokenizer))
model.config.bos_token_id = tokenizer.bos_token_id
model.config.eos_token_id = tokenizer.eos_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

args_kwargs = {
    "output_dir": str(MODEL_OUT_DIR),
    "overwrite_output_dir": True,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "max_steps": DRY_RUN_STEPS if DRY_RUN_STEPS > 0 else -1,
    "learning_rate": LEARNING_RATE,
    "warmup_ratio": WARMUP_RATIO,
    "weight_decay": WEIGHT_DECAY,
    "per_device_train_batch_size": PER_DEVICE_BATCH,
    "per_device_eval_batch_size": PER_DEVICE_BATCH,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "logging_steps": LOGGING_STEPS,
    "eval_steps": EVAL_STEPS,
    "save_steps": SAVE_STEPS,
    "save_strategy": "steps",
    "save_total_limit": 2,
    "fp16": FP16,
    "bf16": BF16,
    "report_to": "none",
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
    "greater_is_better": False,
    "dataloader_num_workers": 0,
    "dataloader_pin_memory": DEVICE == "cuda",
}

args_params = inspect.signature(TrainingArguments).parameters
if "eval_strategy" in args_params:
    args_kwargs["eval_strategy"] = "steps"
else:
    args_kwargs["evaluation_strategy"] = "steps"

training_args = TrainingArguments(**{k: v for k, v in args_kwargs.items() if k in args_params})

if DRY_RUN_STEPS > 0:
    print(f"DRY RUN MODE: training will stop after {DRY_RUN_STEPS} step(s). Set DRY_RUN_STEPS=0 for full CPT.")

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": train_ds,
    "eval_dataset": val_ds,
    "data_collator": collator,
}
trainer_params = inspect.signature(Trainer).parameters
if "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

print("precision:", PRECISION_MODE, "bf16=", BF16, "fp16=", FP16)
trainer = Trainer(**trainer_kwargs)
trainer


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


precision: bf16 bf16= True fp16= False


In [24]:
train_output = trainer.train()

if DRY_RUN_STEPS > 0:
    result = {
        "dry_run": True,
        "train_output": train_output.metrics,
        "mixture_stats": payload["mixture_stats"],
        "note": "Dry run finished. Set DRY_RUN_STEPS=0 and rerun training for the full CPT checkpoint.",
    }
else:
    trainer.save_model(str(FINAL_MODEL_DIR))
    tokenizer.save_pretrained(str(FINAL_MODEL_DIR))

    result = {
        "dry_run": False,
        "train_output": train_output.metrics,
        "mixture_stats": payload["mixture_stats"],
        "final_model_dir": str(FINAL_MODEL_DIR),
    }
    with (ARTIFACT_DIR / "train_result.json").open("w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

result

Step,Training Loss,Validation Loss
200,3.041200,2.824227
400,2.350900,2.257581
600,2.186200,2.070004
800,2.054200,1.973888


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


{'dry_run': False,
 'train_output': {'train_runtime': 409.1658,
  'train_samples_per_second': 71.854,
  'train_steps_per_second': 2.244,
  'total_flos': 5107593747456000.0,
  'train_loss': 2.8744975486871724,
  'epoch': 2.9975510204081632},
 'mixture_stats': {'total': 10000,
  'train': 9800,
  'validation': 200,
  'behavior': 5000,
  'metadata_total': 5000,
  'metadata_compact': 3500,
  'metadata_description': 1500,
  'max_seq_length': 384,
  'history_last_k': 48,
  'description_mode': 'short',
  'desc_text_max_tokens': 96},
 'final_model_dir': 'C:\\Users\\User\\plum-ml1m-repro\\data\\processed\\artifacts\\cpt_gpt2_small_sid_v2_qwen4b_plum_m50_b50_descshort\\final'}

## 10. After CPT

Use `FINAL_MODEL_DIR` as the CPT-initialized GPT2-small checkpoint for the next SFT notebook. The important artifact is:

`data/processed/artifacts/cpt_gpt2_small_sid_v2_qwen4b_plum_m50_b50_descshort/final`


In [25]:
ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

MODEL_DIR = ROOT / "data/processed/artifacts/cpt_gpt2_small_sid_v2_qwen4b_plum_m50_b50_descshort/final"
ITEM_PROFILE_PATH = ROOT / "data/processed/item_features/qwen4b_audited_v1_item_profiles.parquet"
SID_PATH = ROOT / "runs/qwen4b_rqvae_sid_v2_plum/SIDs_best.npy"
TRAIN_PATH = ROOT / "data/processed/splits/train.parquet"
USERS_PATH = ROOT / "data/raw/ml-1m/users.dat"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = GPT2TokenizerFast.from_pretrained(MODEL_DIR)
model = GPT2LMHeadModel.from_pretrained(
    MODEL_DIR,
    torch_dtype=torch.bfloat16 if DEVICE == "cuda" and torch.cuda.is_bf16_supported() else torch.float16 if DEVICE == "cuda" else torch.float32,
).to(DEVICE)
model.eval()

profiles = pd.read_parquet(ITEM_PROFILE_PATH).sort_values("item_idx").reset_index(drop=True)
sids = np.load(SID_PATH)

print(DEVICE, MODEL_DIR)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


cuda C:\Users\User\plum-ml1m-repro\data\processed\artifacts\cpt_gpt2_small_sid_v2_qwen4b_plum_m50_b50_descshort\final


In [27]:
def generate(
    prompt,
    max_new_tokens=80,
    do_sample=False,
    temperature=0.8,
    top_p=0.9,
    num_beams=1,
):
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature if do_sample else None,
            top_p=top_p if do_sample else None,
            num_beams=num_beams,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.08,
        )

    return tokenizer.decode(out[0], skip_special_tokens=False)


def sid_text(item_idx):
    sid = sids[int(item_idx)]
    return " ".join(f"<sid_{level}_{int(code)}>" for level, code in enumerate(sid))


def clean_generated(text):
    return text.replace("<bos>", "\n<bos>").replace("<e>", "\n<e>").replace("</e>", "</e>\n")

In [28]:
item_idx = 0  # Toy Story
row = profiles.iloc[item_idx]

prompt = f"<bos> <meta> Movie {sid_text(item_idx)} has title:"
print("GROUND TRUTH:")
print(row[["title", "year", "genres_text"]])
print("\nPROMPT:")
print(prompt)

print("\nGREEDY:")
print(generate(prompt, max_new_tokens=70, do_sample=False))

print("\nSAMPLED:")
print(generate(prompt, max_new_tokens=70, do_sample=True, temperature=0.8, top_p=0.9))


GROUND TRUTH:
title                       Toy Story (1995)
year                                    1995
genres_text    Animation, Children's, Comedy
Name: 0, dtype: object

PROMPT:
<bos> <meta> Movie <sid_0_32> <sid_1_134> <sid_2_20> <sid_3_18> has title:

GREEDY:
<bos> <meta> Movie <sid_0_32> <sid_1_134> <sid_2_20> <sid_3_18> has title: The Last Man on Earth. Release year: 1999, Genres: Action-Adventure/Fantasy (Novel), SciFi and Thriller; Westerns in the 1990's with a touch of horror at times.."Genre : Adventure". </meta> <eos>

SAMPLED:
<bos> <meta> Movie <sid_0_32> <sid_1_134> <sid_2_20> <sid_3_18> has title: Baby and the Beanstalk. Release year, 1995; Genres Crime Drama II-Noir/ SciFi Thriller III (Lentil), Horror Story Trivia with Plot Summary by Alex Mitzel & Dave Dube of Los Angeles Film Festival 2nd annual award winning Linnitese Theatre Awards. Documentary about film festivals in


In [29]:
for item_idx in [0, 1, 1104, 1178, 257, 1192]:
    row = profiles.iloc[item_idx]
    prompt = f"<bos> <meta> Movie {sid_text(item_idx)} has title:"
    
    print("=" * 100)
    print("ITEM:", item_idx, "|", row["title"], "|", row["genres_text"])
    print(generate(prompt, max_new_tokens=60, do_sample=False))

ITEM: 0 | Toy Story (1995) | Animation, Children's, Comedy
<bos> <meta> Movie <sid_0_32> <sid_1_134> <sid_2_20> <sid_3_18> has title: The Last Man on Earth. Release year: 1999, Genres: Action-Adventure/Fantasy (Novel), SciFi and Thriller; Westerns in the 1990's with a touch of horror at times.."Genre : Adventure". </meta> <eos>
ITEM: 1 | Jumanji (1995) | Adventure, Children's, Fantasy
<bos> <meta> Movie <sid_0_469> <sid_1_18> <sid_2_69> <sid_3_31> has title: The Last Man on Earth. Release year, 1996; Genres : Action-Adventure/Fantasy (Novel), SciFi and Thriller of the Apocalypse 2nd person action adventure set in a post apocalyptic world where humans are stranded after an asteroid crashes into their planet's atmosphere causing massive destruction to
ITEM: 1104 | One Flew Over the Cuckoo's Nest (1975) | Drama
<bos> <meta> Movie <sid_0_227> <sid_1_113> <sid_2_59> <sid_3_30> has title: The Last Man on Earth. Release year: 1999, Genres: Action-Adventure and SciFi/ Thriller; Westerns (Noir)

In [30]:
for item_idx in [0, 1, 1104, 1178, 257]:
    row = profiles.iloc[item_idx]
    prompt = f"<bos> <desc> Movie {sid_text(item_idx)} plot overview:"
    
    print("=" * 100)
    print("ITEM:", item_idx, "|", row["title"])
    print("TRUE:", row["description_text"][:250])
    print("\nGEN:")
    print(generate(prompt, max_new_tokens=100, do_sample=True, temperature=0.85, top_p=0.92))

ITEM: 0 | Toy Story (1995)
TRUE: Plot overview: Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from th

GEN:
<bos> <desc> Movie <sid_0_32> <sid_1_134> <sid_2_20> <sid_3_18> plot overview: Plot summary. A small town is torn apart by a ferocious criminal group, and the city itself falls into chaos as members of an elite military team seek to stop them from getting away with their crimes against humanity during World War II." Release year: 1963 Genres: Action/Adventure (Fiore).
<eos>
ITEM: 1 | Jumanji (1995)
TRUE: Plot overview: When siblings Judy and Peter discover an enchanted board game that opens the door to a magical world, they unwittingly invite Alan -- an adult who's been trapped inside the game for 26 years -- into their living room. Alan's only hope 

GEN:
<bos> <desc> Movie <sid_0_469> <sid_1_18> <si

In [31]:
train = pd.read_parquet(TRAIN_PATH)
users = pd.read_csv(
    USERS_PATH,
    sep="::",
    engine="python",
    names=["user_id", "gender", "age", "occupation", "zip"],
    encoding="latin-1",
)

users_by_id = users.set_index("user_id", drop=False)

def rating_token(rating):
    return f"<rat_{int(rating)}>"

def user_tokens(user_row):
    return f"<gen_{user_row.gender}> <age_{int(user_row.age)}> <occ_{int(user_row.occupation)}>"

def event_text(item_idx, rating=None):
    text = f"<e> {sid_text(item_idx)}"
    if rating is not None:
        text += f" {rating_token(rating)}"
    text += " </e>"
    return text

def make_history_prompt(user_id, last_k=8):
    user_row = users_by_id.loc[user_id]
    hist = (
        train[train["user_id"] == user_id]
        .sort_values("timestamp")
        .tail(last_k)
    )
    
    events = " ".join(
        event_text(row.item_idx, row.rating)
        for row in hist.itertuples(index=False)
    )
    
    return f"<bos> <user> {user_tokens(user_row)} </user> <hist> {events}"

prompt = make_history_prompt(user_id=1, last_k=8)
print(clean_generated(prompt))


<bos> <user> <gen_F> <age_1> <occ_10> </user> <hist> 
<e> <sid_0_469> <sid_1_139> <sid_2_93> <sid_3_23> <rat_3> </e>
 
<e> <sid_0_32> <sid_1_134> <sid_2_109> <sid_3_52> <rat_3> </e>
 
<e> <sid_0_469> <sid_1_77> <sid_2_84> <sid_3_33> <rat_4> </e>
 
<e> <sid_0_32> <sid_1_134> <sid_2_20> <sid_3_18> <rat_5> </e>
 
<e> <sid_0_469> <sid_1_138> <sid_2_89> <sid_3_48> <rat_4> </e>
 
<e> <sid_0_32> <sid_1_216> <sid_2_30> <sid_3_26> <rat_5> </e>
 
<e> <sid_0_469> <sid_1_7> <sid_2_49> <sid_3_26> <rat_4> </e>
 
<e> <sid_0_469> <sid_1_102> <sid_2_41> <sid_3_49> <rat_4> </e>



In [33]:
import re

In [34]:
SID_RE = re.compile(r"<sid_(\d+)_(\d+)>")

def extract_sid_sequences(text):
    tokens = SID_RE.findall(text)
    tokens = [(int(level), int(code)) for level, code in tokens]
    
    sequences = []
    cur = []
    for level, code in tokens:
        if level == 0 and cur:
            sequences.append(cur)
            cur = []
        cur.append((level, code))
        if len(cur) == 4:
            sequences.append(cur)
            cur = []
    return sequences

out = generate(make_history_prompt(user_id=1, last_k=8), max_new_tokens=100, do_sample=True)
print(clean_generated(out))
print(extract_sid_sequences(out))


<bos> <user> <gen_F> <age_1> <occ_10> </user> <hist> 
<e> <sid_0_469> <sid_1_139> <sid_2_93> <sid_3_23> <rat_3> </e>
 
<e> <sid_0_32> <sid_1_134> <sid_2_109> <sid_3_52> <rat_3> </e>
 
<e> <sid_0_469> <sid_1_77> <sid_2_84> <sid_3_33> <rat_4> </e>
 
<e> <sid_0_32> <sid_1_134> <sid_2_20> <sid_3_18> <rat_5> </e>
 
<e> <sid_0_469> <sid_1_138> <sid_2_89> <sid_3_48> <rat_4> </e>
 
<e> <sid_0_32> <sid_1_216> <sid_2_30> <sid_3_26> <rat_5> </e>
 
<e> <sid_0_469> <sid_1_7> <sid_2_49> <sid_3_26> <rat_4> </e>
 
<e> <sid_0_469> <sid_1_102> <sid_2_41> <sid_3_49> <rat_4> </e>
 has title: Kiss. Release year (MM). Genres/ Comedy, Drama and Romance in Western Animation or Musical 3D-Noir II of The Secret Life for Children 2nd Century Bicentennial Sci Fi Movie III on Broadway. A production with a plot that takes place after the fall from grace and return to Earth during World War I as part time soldiers who survived under cover against German invasion by American troops in Africa before being captured ag